In [0]:
%run ../config/config

In [0]:
from pyspark.sql import functions as F
from pathlib import Path
from delta.tables import DeltaTable
import sys

SRC_PATH = str(Path.cwd().parent / "src")
sys.path.insert(0, SRC_PATH)

from silver import prepare_silver_dataframe, apply_scd_type_2

spark.conf.set("spark.databricks.execution.timeout", "18000")

BRONZE_CURRENT_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_CURRENT_TABLE}"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
CHECKPOINT_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{CDF_CHECKPOINT}"

PIPELINE_NAME = "openfoodfacts_silver"

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHECKPOINT_TABLE} (
        pipeline_name STRING,
        source_table STRING,
        last_processed_version BIGINT,
        updated_at TIMESTAMP
    )
    USING DELTA
""")

DataFrame[]

In [0]:
def update_checkpoint(version):
    spark.sql(f"""
        UPDATE {CHECKPOINT_TABLE}
        SET
            last_processed_version = {version},
            updated_at = current_timestamp()
        WHERE pipeline_name = '{PIPELINE_NAME}'
    """)

    print(f"Checkpoint advanced to version {version}")

In [0]:
latest_version = (
    DeltaTable.forName(spark, BRONZE_CURRENT_TABLE)
    .history(1)
    .select("version")
    .first()["version"]
)

print(f"Latest Bronze version: {latest_version}")

checkpoint_row = (
    spark.table(CHECKPOINT_TABLE)
    .filter(F.col("pipeline_name") == PIPELINE_NAME)
    .first()
)

if checkpoint_row is None:
    print("No checkpoint found. Performing initial Silver load.")
    bootstrap_version = latest_version

    initial_bronze_df = (
        spark.read
        .format("delta")
        .option("versionAsOf", bootstrap_version)
        .table(BRONZE_CURRENT_TABLE)
    )

    print(f"Bootstrapping from Bronze version {bootstrap_version}")

    initial_silver_df = prepare_silver_dataframe(
        bronze_snapshot_df=initial_bronze_df,
        columns_to_keep=COLUMNS_TO_KEEP,
        rename_map=RENAME_MAP,
        pnn_categories=PNN_CATEGORIES,
        microelement_columns=MICROELEMENT_COLS,
        country_map=COUNTRY_MAP,
    )

    apply_scd_type_2(
        spark=spark,
        source_df=initial_silver_df,
        target_table_path=SILVER_TABLE,
        business_key=BUSINESS_KEY,
        columns_to_keep=COLUMNS_TO_KEEP,
        rename_map=RENAME_MAP,
    )

    spark.sql(f"""
        INSERT INTO {CHECKPOINT_TABLE}
        VALUES (
            '{PIPELINE_NAME}',
            '{BRONZE_CURRENT_TABLE}',
            {bootstrap_version},
            current_timestamp()
        )
    """)

    print(
        f"Initial Silver load completed. "
        f"Checkpoint initialized to version {bootstrap_version}."
    )

else:
    last_processed_version = checkpoint_row["last_processed_version"]
    print(f"Last processed Bronze version: {last_processed_version}")

    start_version = last_processed_version + 1
    end_version = latest_version

    if start_version > end_version:
        print("Silver is already up to date. No new Bronze versions")

    else:
        print(
            f"Processing Bronze versions "
            f"{start_version} to {end_version}"
        )

        for version in range(start_version, end_version + 1):
            print(f"Processing Bronze version {version}")

            cdf_df = (
                spark.read
                .format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", version)
                .option("endingVersion", version)
                .table(BRONZE_CURRENT_TABLE)
            )

            inserts_updates_df = cdf_df.filter(
                F.col("_change_type").isin(
                    "insert",
                    "update_postimage"
                )
            )

            if inserts_updates_df.limit(1).count() > 0:
                silver_source_df = prepare_silver_dataframe(
                    bronze_snapshot_df=inserts_updates_df,
                    columns_to_keep=COLUMNS_TO_KEEP,
                    rename_map=RENAME_MAP,
                    pnn_categories=PNN_CATEGORIES,
                    microelement_columns=MICROELEMENT_COLS,
                    country_map=COUNTRY_MAP,
                )

                apply_scd_type_2(
                    spark=spark,
                    source_df=silver_source_df,
                    target_table_path=SILVER_TABLE,
                    business_key=BUSINESS_KEY,
                    columns_to_keep=COLUMNS_TO_KEEP,
                    rename_map=RENAME_MAP,
                )

            else:
                print(f"No insert/update events in version {version}.")

            update_checkpoint(version)

    print("Silver CDF processing completed.")

spark.table(CHECKPOINT_TABLE).filter(
    F.col("pipeline_name") == PIPELINE_NAME
).show(truncate=False)


Latest Bronze version: 7
Last processed Bronze version: 6
Processing Bronze versions 7 to 7
Processing Bronze version 7
Checkpoint advanced to version 7
Silver CDF processing completed.
+--------------------+-------------------------------------------------------------+----------------------+--------------------------+
|pipeline_name       |source_table                                                 |last_processed_version|updated_at                |
+--------------------+-------------------------------------------------------------+----------------------+--------------------------+
|openfoodfacts_silver|dbr_dev_ua5816bd.gulidov_artem_bronze.products_bronze_current|7                     |2026-11-18 15:15:44.296355|
+--------------------+-------------------------------------------------------------+----------------------+--------------------------+

